# Vector RAG Pipeline — FinanceBench (full batch run)

**Paper:** Kim et al. (2025), _GAR: Generative Answer Refinement for Financial QA_, arXiv 2503.15191
**Thesis:** Vector RAG vs. Vectorless RAG vs. Long-Context LLMs on FinanceBench

---

## What changed from the single-question version

The earlier two-notebook setup (`vector_rag_pipeline.ipynb` for Colab-only indexing,
`vector_rag_pipeline_local.ipynb` for the rest) existed because Stella 1.5B couldn't
load locally, so query vectors had to be embedded in Colab and hand-carried to a
local kernel one `.npy` file at a time. Now that this notebook runs directly against
a Colab-backed kernel from VS Code, that split is gone — everything below runs in one
kernel session, and there's no more manual file hand-off.

All the actual logic (chunking, indexing, retrieval, generation) now lives in
importable modules under `pipelines/vector_rag/`, not in these cells — this notebook
just calls them in order. That's what makes a 150-question × 84-document batch run
practical instead of copy-pasting cells 150 times: the same functions that were
proven correct on one question are reused for all of them.

```
Stage 0  Setup            — Drive mount, repo clone/pull, API keys, cost tracker
Stage 1  Load data        — 150 questions, 84 unique documents
Stage 2  Load models      — Stella tokenizer + embedding model (once)
Stage 3  Index ALL docs   — chunk + embed every document (resumable, GPU)
Stage 4  Embed ALL queries— expand + embed every question's query (resumable, GPU)
Stage 5  Run ALL questions— hybrid retrieve → rerank → select → generate → score
Stage 6  Summarize        — answer quality, retrieval Recall/MRR, latency, tokens
```

### Resumability — important for a run this long

Stages 3-5 are each **resumable**: every document / query / question that already has
saved output on disk is skipped, and only new work is done. If the Colab session
disconnects partway through (indexing all 84 docs is GPU-bound and can take a few
hours), just re-run the same cell — it picks up where it left off instead of
restarting from zero. Failures are caught per-item (one bad PDF or one malformed API
response doesn't abort the whole batch) and logged to `.log` files next to the
output, which are worth checking after a run finishes.

### Known scope decisions for this pass (see chat for the reasoning)
- **Gemini only.** DeepSeek V4 isn't wired in yet — `generation_model` is passed
  explicitly everywhere so adding it later is a config change, not a rewrite.
- **One run per question**, not the "median of several runs" the project brief
  specifies for latency. Fine for getting answer-quality and retrieval numbers now;
  revisit before those numbers go in the thesis.


---
## Stage 0 — Setup

Drive mount, repo clone/pull, API keys, cost tracker. Same as the original Colab notebook's Stage 0.

In [2]:
import os, sys, json, time, textwrap
from pathlib import Path
import pandas as pd
from dotenv import load_dotenv
from google.colab import drive

drive.mount('/content/drive')

REPO_ROOT = Path('/content/drive/MyDrive/financebench_project')
if not (REPO_ROOT / "data" / "financebench_open_source.jsonl").exists():
    print(f"Repo not found at {REPO_ROOT} — cloning ...")
    !git clone https://github.com/shaliqsv/financebench-rag-thesis.git "{REPO_ROOT}"
else:
    # Plain `git pull` fails silently-ish ("You are not currently on a branch")
    # if this clone ever ends up in a detached-HEAD state (e.g. an interrupted
    # checkout) — it fetches fine but then has nothing to merge into. Checking
    # out main explicitly every time makes this self-healing instead of a
    # manual fix you have to remember to run.
    print(f"Repo already present at {REPO_ROOT} — syncing to latest main ...")
    !git -C "{REPO_ROOT}" fetch origin
    !git -C "{REPO_ROOT}" checkout main
    !git -C "{REPO_ROOT}" pull origin main

DATA_DIR  = REPO_ROOT / "data"
PDF_DIR   = REPO_ROOT / "pdfs"
INDEX_DIR = REPO_ROOT / "experiments" / "results" / "vector_rag_index"
QUERIES_DIR = INDEX_DIR / "queries"
RESULTS_PATH = REPO_ROOT / "experiments" / "results" / "vector_rag_results.jsonl"

sys.path.insert(0, str(REPO_ROOT))
print(f"REPO_ROOT: {REPO_ROOT}")

load_dotenv(REPO_ROOT / ".env", override=True)
GOOGLE_API_KEY   = os.getenv("GOOGLE_API_KEY", "")
DEEPSEEK_API_KEY = os.getenv("DEEPSEEK_API_KEY", "")
GROQ_API_KEY     = os.getenv("GROQ_API_KEY", "")
VOYAGE_API_KEY   = os.getenv("VOYAGE_API_KEY", "")
GH_TOKEN         = os.getenv("GH_TOKEN", "")

print("\nAPI keys:")
for name, val in [("GOOGLE_API_KEY", GOOGLE_API_KEY), ("DEEPSEEK_API_KEY", DEEPSEEK_API_KEY),
                  ("GROQ_API_KEY", GROQ_API_KEY), ("VOYAGE_API_KEY", VOYAGE_API_KEY),
                  ("GH_TOKEN", GH_TOKEN)]:
    print(f"  {name:<20}: {'ok' if val else 'MISSING - fill in .env'}")

Mounted at /content/drive
Repo already present at /content/drive/MyDrive/financebench_project — syncing to latest main ...
Already on 'main'
Your branch is ahead of 'origin/main' by 1 commit.
  (use "git push" to publish your local commits)
From https://github.com/shaliqsv/financebench-rag-thesis
 * branch            main       -> FETCH_HEAD
Already up to date.
REPO_ROOT: /content/drive/MyDrive/financebench_project

API keys:
  GOOGLE_API_KEY      : ok
  DEEPSEEK_API_KEY    : MISSING - fill in .env
  GROQ_API_KEY        : ok
  VOYAGE_API_KEY      : ok
  GH_TOKEN            : MISSING - fill in .env


### Git identity + push auth for Colab

The "sync to GitHub" cells below (after Stages 3/4/5) run inside the Colab
container, which has no git identity and no saved GitHub credentials. Without
this cell, `git commit` fails with "Author identity unknown" and `git push`
fails with "could not read Username" — and this failure is easy to miss, since
the sync cells swallow the commit error with `|| echo "(nothing new to
commit)"`. That's exactly what happened on the last full Stage 3 run: all 84
documents were indexed correctly on Drive, but none of it reached GitHub.

Run this cell once per Colab session, right after Stage 0, before Stage 3.
Requires `GH_TOKEN` in `.env` (a GitHub personal access token, repo scope —
see `.env.example`).

In [3]:
!git -C "{REPO_ROOT}" config user.email "shaliqv25@gmail.com"
!git -C "{REPO_ROOT}" config user.name "shaliqsv"

if GH_TOKEN:
    !git -C "{REPO_ROOT}" remote set-url origin https://{GH_TOKEN}@github.com/shaliqsv/financebench-rag-thesis.git
    print("git identity set, remote configured with token auth")
else:
    print("WARNING: GH_TOKEN missing from .env — sync-to-GitHub cells will fail to push. "
          "See .env.example for how to create one.")

In [4]:
import subprocess

# Shared by the Stage 3/4/5 sync-during-the-run calls below. Small, frequent syncs
# (rather than one sync at the very end) mean a dropped Colab/VS Code connection on a
# slow or unstable line loses at most one batch's worth of work, not the whole run.
def _git_sync(paths: list[str], commit_msg: str):
    subprocess.run(["git", "-C", str(REPO_ROOT), "add", *paths], check=True)
    commit = subprocess.run(
        ["git", "-C", str(REPO_ROOT), "commit", "-m", commit_msg],
        capture_output=True, text=True,
    )
    if commit.returncode != 0:
        print("    (nothing new to commit)")
    subprocess.run(["git", "-C", str(REPO_ROOT), "push", "origin", "main"], check=True)

def sync_index_to_github():
    _git_sync(["experiments/results/vector_rag_index"], "Sync indexed documents / query embeddings")

def sync_results_to_github():
    _git_sync(
        ["experiments/results/vector_rag_results.jsonl", "experiments/results/vector_rag_costs.jsonl"],
        "Sync results + cost log",
    )


In [5]:
%pip install -q pymupdf4llm tiktoken sentence-transformers rank_bm25 pyarrow "transformers==4.51.3" "sentence-transformers==3.3.1" google-genai voyageai groq

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 268.8/268.8 kB 9.6 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 10.4/10.4 MB 108.6 MB/s eta 0:00:0000:010:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 176.4/176.4 kB 19.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 25.8/25.8 MB 83.5 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 42.9/42.9 MB 23.4 MB/s eta 0:00:00:00:0100:01
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 46.6/46.6 kB 4.8 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 143.7/143.7 kB 15.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 566.4/566.4 kB 48.2 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 3.1/3.1 MB 110.4 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 23.1/23.1 MB 85.5 MB/s eta 0:00:00:00:0100:01
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependenc

In [6]:
from evaluation.cost_tracker import CostTracker
from groq import Groq
from google import genai
import voyageai

cost_tracker = CostTracker(REPO_ROOT / "experiments" / "results" / "vector_rag_costs.jsonl")
print(f"Logging costs to {cost_tracker.log_path}")

GENERATION_MODEL = "gemini-3.5-flash"   # gemini-2.5-flash was deprecated for new API keys/projects
JUDGE_MODEL = "openai/gpt-oss-120b"     # via Groq free tier — see CLAUDE.md for why

genai_client   = genai.Client(api_key=GOOGLE_API_KEY)
voyage_client  = voyageai.Client(api_key=VOYAGE_API_KEY)
judge_client   = Groq(api_key=GROQ_API_KEY) if GROQ_API_KEY else None

print(f"Generation model: {GENERATION_MODEL}")
print(f"Judge model:      {JUDGE_MODEL}")

Logging costs to /content/drive/MyDrive/financebench_project/experiments/results/vector_rag_costs.jsonl
Generation model: gemini-3.5-flash
Judge model:      openai/gpt-oss-120b


---
## Stage 1 — Load FinanceBench data

All 150 questions across 84 unique source documents (some documents have multiple questions).

In [7]:
df_questions = pd.read_json(DATA_DIR / "financebench_open_source.jsonl", lines=True)
df_meta      = pd.read_json(DATA_DIR / "financebench_document_information.jsonl", lines=True)
df           = pd.merge(df_questions, df_meta, on=["doc_name", "company"])

doc_names = sorted(df.doc_name.unique())

print(f"Total questions : {len(df)}")
print(f"Unique documents: {len(doc_names)}")
df[["financebench_id", "doc_name", "question"]].head(3)

Total questions : 150
Unique documents: 84


,financebench_id,doc_name,question
0,financebench_id_03029,3M_2018_10K,What is the FY2018 capital expenditure amount ...
1,financebench_id_04672,3M_2018_10K,Assume that you are a public equities analyst....
2,financebench_id_00499,3M_2022_10K,Is 3M a capital-intensive business based on FY...


---
## Stage 2 — Load Stella (tokenizer + embedding model)

Loaded once here and passed into every batch call below, instead of being reloaded
per document/query — reloading a 1.5B-parameter model 84+150 times would dominate
the runtime for no benefit.

In [8]:
from transformers import AutoTokenizer

STELLA_MODEL = "NovaSearch/stella_en_1.5B_v5"
print(f"Loading tokenizer for {STELLA_MODEL} ...")
_stella_tokenizer = AutoTokenizer.from_pretrained(STELLA_MODEL, trust_remote_code=True)

def count_tokens(text: str) -> int:
    return len(_stella_tokenizer.encode(text, add_special_tokens=False))

print("Tokenizer ready")

Loading tokenizer for NovaSearch/stella_en_1.5B_v5 ...


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


tokenizer_config.json: 0.00B [00:00, ?B/s]

tokenization_qwen.py: 0.00B [00:00, ?B/s]

A new version of the following files was downloaded from https://huggingface.co/NovaSearch/stella_en_1.5B_v5:
- tokenization_qwen.py
. Make sure to double-check they do not contain any added malicious code. To avoid downloading new versions of the code file, you can pin a revision.


...

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

added_tokens.json:   0%|          | 0.00/80.0 [00:00<?, ?B/s]

special_tokens_map.json:   0%|          | 0.00/370 [00:00<?, ?B/s]

Tokenizer ready


In [9]:
import torch
from sentence_transformers import SentenceTransformer

device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Device: {device}")
if device == "cpu":
    print("No GPU detected — indexing 84 documents on CPU will be very slow. "
          "In Colab: Runtime -> Change runtime type -> T4 GPU.")

t0 = time.time()
embed_model = SentenceTransformer(
    STELLA_MODEL,
    trust_remote_code=True,
    device=device,
    config_kwargs={"use_memory_efficient_attention": False, "unpad_inputs": False},
)
print(f"Loaded in {time.time() - t0:.1f}s — embedding dim {embed_model.get_sentence_embedding_dimension()}")

Device: cuda


modules.json:   0%|          | 0.00/316 [00:00<?, ?B/s]

config_sentence_transformers.json:   0%|          | 0.00/397 [00:00<?, ?B/s]

README.md: 0.00B [00:00, ?B/s]

sentence_bert_config.json:   0%|          | 0.00/51.0 [00:00<?, ?B/s]

...

config.json:   0%|          | 0.00/289 [00:00<?, ?B/s]

2_Dense_1024/model.safetensors:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

config.json:   0%|          | 0.00/122 [00:00<?, ?B/s]

2_Dense_1024/pytorch_model.bin:   0%|          | 0.00/6.30M [00:00<?, ?B/s]

Loaded in 96.7s — embedding dim 1024


---
## Stage 3 — Index every document

Chunk (page-bounded, ≤512 tokens) + embed (Stella) every one of the 84 unique
documents, saving `{doc_name}_chunks.parquet` and `{doc_name}_dense.npy` to
`INDEX_DIR`. **Resumable** — a document already indexed is skipped, so re-running
this cell after an interruption only does the remaining work.

This is the slow part of the whole pipeline: on the first test document (160 pages),
parsing took ~150s and embedding ~145s, so budget roughly 5 minutes/document ×
84 documents ≈ several hours for a from-scratch run. Safe to leave running
unattended and re-run if the session drops.

**Auto-syncs to GitHub every 5 newly-indexed docs** (`sync_every=5` below) instead of
only once at the end — on a slow/unstable connection this caps how much indexing
work a dropped session can lose to ~25 minutes' worth, not the whole run.

In [16]:
import importlib
import pipelines.vector_rag.indexing
importlib.reload(pipelines.vector_rag.indexing)
from pipelines.vector_rag.indexing import index_all_documents


index_results = index_all_documents(
    doc_names=doc_names,
    pdf_dir=PDF_DIR,
    embed_model=embed_model,
    count_tokens=count_tokens,
    index_dir=INDEX_DIR,
    sync_every=5,
    sync_fn=sync_index_to_github,
)


TypeError: index_all_documents() got an unexpected keyword argument 'sync_every'

In [10]:
index_results

{'indexed': [],
 'skipped': ['3M_2018_10K',
  '3M_2022_10K',
  '3M_2023Q2_10Q',
  'ACTIVISIONBLIZZARD_2019_10K',
  'ADOBE_2015_10K',
  'ADOBE_2016_10K',
  'ADOBE_2017_10K',
  'ADOBE_2022_10K',
  'AES_2022_10K',
  'AMAZON_2017_10K',
  'AMAZON_2019_10K',
  'AMCOR_2020_10K',
  'AMCOR_2022_8K_dated-2022-07-01',
  'AMCOR_2023Q2_10Q',
  'AMCOR_2023Q4_EARNINGS',
  'AMCOR_2023_10K',
  'AMD_2015_10K',
  'AMD_2022_10K',
  'AMERICANEXPRESS_2022_10K',
  'AMERICANWATERWORKS_2020_10K',
  'AMERICANWATERWORKS_2021_10K',
  'AMERICANWATERWORKS_2022_10K',
  'BESTBUY_2017_10K',
  'BESTBUY_2019_10K',
  'BESTBUY_2023_10K',
  'BESTBUY_2024Q2_10Q',
  'BLOCK_2016_10K',
  'BLOCK_2020_10K',
  'BOEING_2018_10K',
  'BOEING_2022_10K',
  'COCACOLA_2017_10K',
  'COCACOLA_2021_10K',
  'COCACOLA_2022_10K',
  'CORNING_2020_10K',
  'CORNING_2021_10K',
  'CORNING_2022_10K',
  'COSTCO_2021_10K',
  'CVSHEALTH_2018_10K',
  'CVSHEALTH_2022_10K',
  'FOOTLOCKER_2022_8K_dated-2022-05-20',
  'FOOTLOCKER_2022_8K_dated_2022-08-19',

### Sync indexed documents back to GitHub

`INDEX_DIR` lives on Drive, which only this Colab session can see. Committing it
to the repo means a plain `git pull` on your **local machine** fetches every
document already indexed here — no GPU, no rerun, no Colab session needed to use
it later (e.g. to inspect chunks, or to run Stage 5 from a local CPU kernel).

The Stage 3 cell above now syncs automatically every 5 docs, so you shouldn't need
this manually — kept here as a one-off you can run any time (e.g. to sync
immediately without waiting for the next automatic batch).

In [ ]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_index
!git -C "{REPO_ROOT}" commit -m "Sync indexed documents" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

[main 4bf33a8] Sync indexed documents
 166 files changed, 0 insertions(+), 0 deletions(-)
 create mode 100644 experiments/results/vector_rag_index/3M_2022_10K_chunks.parquet
 create mode 100644 experiments/results/vector_rag_index/3M_2022_10K_dense.npy
 create mode 100644 experiments/results/vector_rag_index/3M_2023Q2_10Q_chunks.parquet
 create mode 100644 experiments/results/vector_rag_index/3M_2023Q2_10Q_dense.npy
 create mode 100644 experiments/results/vector_rag_index/ACTIVISIONBLIZZARD_2019_10K_chunks.parquet
 create mode 100644 experiments/results/vector_rag_index/ACTIVISIONBLIZZARD_2019_10K_dense.npy
 create mode 100644 experiments/results/vector_rag_index/ADOBE_2015_10K_chunks.parquet
 create mode 100644 experiments/results/vector_rag_index/ADOBE_2015_10K_dense.npy
 create mode 100644 experiments/results/vector_rag_index/ADOBE_2016_10K_chunks.parquet
 create mode 100644 experiments/results/vector_rag_index/ADOBE_2016_10K_dense.npy
 create mode 100644 experiments/results/vector_

---
## Stage 4 — Expand + embed every question's query

For each of the 150 questions: ask Gemini to expand the question into a retrieval-
friendly query (Stage 4 from the single-question notebook), then embed that expanded
query with Stella. Saves `{financebench_id}__expanded.npy` / `.json` to
`QUERIES_DIR`. **Resumable** the same way as Stage 3, and **auto-syncs to GitHub
every 20 newly-embedded queries** for the same dropped-connection reasons.

This still needs the GPU-backed Stella model, which is why it runs here rather than
in Stage 5 below.

In [ ]:
from pipelines.vector_rag.batch_embed_queries import embed_all_queries

query_embed_results = embed_all_queries(
    df=df,
    genai_client=genai_client,
    embed_model=embed_model,
    expansion_model=GENERATION_MODEL,
    queries_dir=QUERIES_DIR,
    cost_tracker=cost_tracker,
    sync_every=20,
    sync_fn=sync_index_to_github,
)


### Sync query embeddings back to GitHub

Same reasoning as after Stage 3 — `QUERIES_DIR` is a subfolder of `INDEX_DIR`, so this
covers the newly embedded queries too. The Stage 4 cell above now syncs automatically
every 20 queries; kept here as a manual one-off trigger.

In [ ]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_index
!git -C "{REPO_ROOT}" commit -m "Sync query embeddings" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

---
## Stage 5 — Run every question end to end

For each question with both a document index (Stage 3) and a query embedding
(Stage 4) available: hybrid retrieve (top 10) → Voyage rerank (top 10) → selection
agent → generate → score against the gold answer. **top_k_hybrid defaults to 10, not
the paper's 20** — the Voyage account has no payment method yet, and a 20-chunk
candidate set routinely exceeds the free tier's ~10K-tokens/minute cap. Bump back to
20 (pass `top_k_hybrid=20` below) once billing is added, and note the change in the
thesis methodology section either way. Appends one result row to
`RESULTS_PATH` per question as soon as it finishes, and **skips any
`financebench_id` already present there** — safe to re-run after an interruption,
and safe to run again later if Stage 3/4 produce more indexed docs or embedded
queries. **Auto-syncs to GitHub every 15 newly-completed questions.**

This stage doesn't need the GPU — if it's ever more convenient, `run_all_questions`
can be called from a plain CPU kernel as long as `INDEX_DIR`/`QUERIES_DIR` are
reachable (e.g. after `git pull` on the Drive clone).

In [ ]:
from pipelines.vector_rag.batch_run import run_all_questions

run_results = run_all_questions(
    df=df,
    index_dir=INDEX_DIR,
    queries_dir=QUERIES_DIR,
    out_path=RESULTS_PATH,
    genai_client=genai_client,
    voyage_client=voyage_client,
    judge_client=judge_client,
    generation_model=GENERATION_MODEL,
    judge_model=JUDGE_MODEL,
    cost_tracker=cost_tracker,
    sync_every=15,
    sync_fn=sync_results_to_github,
)


### Sync results + cost log back to GitHub

`RESULTS_PATH` and the cost log grow one line per question. The Stage 5 cell above
now syncs automatically every 15 questions; kept here as a manual one-off trigger
(e.g. to check progress from your local machine right now without waiting).

In [ ]:
!git -C "{REPO_ROOT}" add experiments/results/vector_rag_results.jsonl experiments/results/vector_rag_costs.jsonl
!git -C "{REPO_ROOT}" commit -m "Sync results + cost log" || echo "(nothing new to commit)"
!git -C "{REPO_ROOT}" push origin main

---
## Stage 6 — Summarize

Answer-quality breakdown, retrieval Recall@k/MRR@k (hybrid vs. reranked), latency, and token totals per stage, computed over whatever's in `RESULTS_PATH` so far — doesn't require the full 150 to be done.

In [ ]:
from pipelines.vector_rag.summarize import summarize_results, print_summary

summary = summarize_results(RESULTS_PATH, cost_log_path=cost_tracker.log_path)
print_summary(summary)

### Next steps
1. Check `INDEX_DIR/indexing_errors.log`, `QUERIES_DIR/embedding_errors.log`, and
   `RESULTS_PATH.parent/run_errors.log` for anything that failed and needs a
   second look.
2. Fill in `PRICING_PER_MILLION_TOKENS` in `evaluation/cost_tracker.py` with
   current published rates — token counts are logged throughout, but `cost_usd`
   stays `None` until those are filled in.
3. Once this is stable, wire in DeepSeek V4 as a second `generation_model` and
   re-run Stage 5 with it (Stage 3/4's indexes and query embeddings are model-
   independent and don't need to be redone).
4. Revisit the "one run per question" latency simplification if the thesis needs
   median-of-N latency per the project brief.